In [1]:
# ============================================================
# 1. Área de Interesse (AOI) da propriedade
# ============================================================

from pathlib import Path
import geopandas as gpd

# Pasta do projeto
PROJETO = Path.cwd().parent

# Caminho do Shapefile do imóvel
shp_imovel = (
    PROJETO
    / "data"
    / "raw"
    / "Area_do_Imovel"
    / "Area_do_Imovel.shp"
)

# Leitura do imóvel
imovel = gpd.read_file(shp_imovel)

# Como os dois registros possuem a mesma geometria,
# usamos apenas um deles como limite da propriedade.
aoi = imovel.geometry.iloc[0]

print("✓ AOI criada")
print("Tipo de geometria:", aoi.geom_type)
print("CRS:", imovel.crs)

✓ AOI criada
Tipo de geometria: Polygon
CRS: EPSG:4674


In [2]:
# ============================================================
# 2. Conexão com Google Earth Engine
# ============================================================

import ee

ee.Initialize()

print("✓ Google Earth Engine conectado")

✓ Google Earth Engine conectado


In [3]:
# ============================================================
# 3. Conversão da AOI para geometria do Google Earth Engine
# ============================================================

# Converte a geometria Shapely para GeoJSON
aoi_geojson = aoi.__geo_interface__

# Converte o GeoJSON para uma geometria do Earth Engine
aoi_ee = ee.Geometry(aoi_geojson)

print("✓ AOI convertida para Google Earth Engine")
print("Tipo:", aoi_ee.type().getInfo())

✓ AOI convertida para Google Earth Engine
Tipo: Polygon


In [4]:
# ============================================================
# 4. Coleção Sentinel-2 
# ============================================================

ano = 2025

colecao_ano = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_ee)
    .filterDate(
        f"{ano}-01-01",
        f"{ano + 1}-01-01"
    )
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE",
            20
        )
    )
)

print("✓ Coleção Sentinel-2 carregada")
print(
    f"✓ Imagens disponíveis em {ano}: "
    f"{colecao_ano.size().getInfo()}"
)

✓ Coleção Sentinel-2 carregada
✓ Imagens disponíveis em 2025: 99


In [5]:
# ============================================================
# 5. Seleção da melhor imagem de cada mês 
#    Critério:
#    1. Cobertura da AOI >= 99,99%
#    2. Menor cobertura de nuvens
# ============================================================

imagens_mensais = []

# Área total da AOI
area_aoi = aoi_ee.area(1)


# ------------------------------------------------------------
# Função para calcular quanto da AOI é coberta pela imagem
# ------------------------------------------------------------

def calcular_cobertura_aoi(imagem):

    intersecao = imagem.geometry().intersection(
        aoi_ee,
        1
    )

    area_intersecao = intersecao.area(1)

    cobertura = (
        area_intersecao
        .divide(area_aoi)
        .multiply(100)
    )

    return imagem.set(
        "COBERTURA_AOI",
        cobertura
    )


# ------------------------------------------------------------
# Seleção mensal
# ------------------------------------------------------------

for mes in range(1, 13):

    inicio = f"{ano}-{mes:02d}-01"

    if mes == 12:
        fim = f"{ano + 1}-01-01"
    else:
        fim = f"{ano}-{mes + 1:02d}-01"


    # --------------------------------------------------------
    # Imagens disponíveis no mês
    # --------------------------------------------------------

    colecao_mes = (
        colecao_ano
        .filterDate(inicio, fim)
        .map(calcular_cobertura_aoi)
    )


    # --------------------------------------------------------
    # Mantém somente imagens que cobrem praticamente
    # toda a AOI
    # --------------------------------------------------------

    colecao_completa = (
        colecao_mes
        .filter(
            ee.Filter.gte(
                "COBERTURA_AOI",
                99.99
            )
        )
        .sort(
            "CLOUDY_PIXEL_PERCENTAGE"
        )
    )


    quantidade = colecao_completa.size().getInfo()


    # --------------------------------------------------------
    # Nenhuma imagem com cobertura suficiente
    # --------------------------------------------------------

    if quantidade == 0:

        quantidade_total = colecao_mes.size().getInfo()

        if quantidade_total == 0:

            print(
                f"⚠️ {mes:02d}/{ano} — "
                f"nenhuma imagem encontrada"
            )

        else:

            print(
                f"⚠️ {mes:02d}/{ano} — "
                f"nenhuma imagem cobre 99,99% da AOI"
            )

        continue


    # --------------------------------------------------------
    # Seleciona a imagem com menor nebulosidade
    # entre as que cobrem toda a AOI
    # --------------------------------------------------------

    imagem_mes = ee.Image(
        colecao_completa.first()
    )


    # --------------------------------------------------------
    # Informações da imagem selecionada
    # --------------------------------------------------------

    data = ee.Date(
        imagem_mes.get("system:time_start")
    ).format("YYYY-MM-dd").getInfo()


    nuvens = imagem_mes.get(
        "CLOUDY_PIXEL_PERCENTAGE"
    ).getInfo()


    cobertura = imagem_mes.get(
        "COBERTURA_AOI"
    ).getInfo()


    imagem_id = imagem_mes.get(
        "PRODUCT_ID"
    ).getInfo()


    tile = imagem_mes.get(
        "MGRS_TILE"
    ).getInfo()


    # --------------------------------------------------------
    # Guarda a imagem
    # --------------------------------------------------------

    imagens_mensais.append(
        {
            "mes": mes,
            "data": data,
            "nuvens": nuvens,
            "cobertura_aoi": cobertura,
            "tile": tile,
            "id": imagem_id,
            "imagem": imagem_mes
        }
    )


    # --------------------------------------------------------
    # Resultado
    # --------------------------------------------------------

    print(
        f"✓ {mes:02d}/{ano} | "
        f"{data} | "
        f"nuvens: {nuvens:.2f}% | "
        f"AOI: {cobertura:.2f}% | "
        f"tile: {tile}"
    )


print(
    f"\n✓ Total de imagens selecionadas: "
    f"{len(imagens_mensais)}"
)

✓ 01/2025 | 2025-01-08 | nuvens: 3.87% | AOI: 100.00% | tile: 22KHV
✓ 02/2025 | 2025-02-17 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 03/2025 | 2025-03-09 | nuvens: 0.01% | AOI: 100.00% | tile: 22KHV
✓ 04/2025 | 2025-04-08 | nuvens: 0.02% | AOI: 100.00% | tile: 22KHV
✓ 05/2025 | 2025-05-13 | nuvens: 8.10% | AOI: 100.00% | tile: 23KKQ
✓ 06/2025 | 2025-06-17 | nuvens: 0.00% | AOI: 100.00% | tile: 23KKQ
✓ 07/2025 | 2025-07-27 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 08/2025 | 2025-08-01 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 09/2025 | 2025-09-15 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 10/2025 | 2025-10-05 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 11/2025 | 2025-11-21 | nuvens: 0.04% | AOI: 100.00% | tile: 22KHV
✓ 12/2025 | 2025-12-04 | nuvens: 1.61% | AOI: 100.00% | tile: 22KHV

✓ Total de imagens selecionadas: 12


In [6]:
# ============================================================
# 6. Cálculo do NDVI para cada imagem selecionada
# ============================================================

ndvi_mensais = []

for item in imagens_mensais:

    imagem = item["imagem"]

    ndvi_img = (
        imagem
        .normalizedDifference(["B8", "B4"])
        .rename("NDVI")
        .clip(aoi_ee)
    )

    ndvi_mensais.append(
        {
            "mes": item["mes"],
            "data": item["data"],
            "nuvens": item["nuvens"],
            "id": item["id"],
            "ndvi": ndvi_img
        }
    )

    print(
        f"✓ NDVI criado: "
        f"{item['data']} | "
        f"{item['nuvens']:.2f}% de nuvens"
    )

print(
    f"\n✓ {len(ndvi_mensais)} produtos NDVI preparados"
)

✓ NDVI criado: 2025-01-08 | 3.87% de nuvens
✓ NDVI criado: 2025-02-17 | 0.00% de nuvens
✓ NDVI criado: 2025-03-09 | 0.01% de nuvens
✓ NDVI criado: 2025-04-08 | 0.02% de nuvens
✓ NDVI criado: 2025-05-13 | 8.10% de nuvens
✓ NDVI criado: 2025-06-17 | 0.00% de nuvens
✓ NDVI criado: 2025-07-27 | 0.00% de nuvens
✓ NDVI criado: 2025-08-01 | 0.00% de nuvens
✓ NDVI criado: 2025-09-15 | 0.00% de nuvens
✓ NDVI criado: 2025-10-05 | 0.00% de nuvens
✓ NDVI criado: 2025-11-21 | 0.04% de nuvens
✓ NDVI criado: 2025-12-04 | 1.61% de nuvens

✓ 12 produtos NDVI preparados


In [7]:
# ============================================================
# 7. Exportação dos NDVI para a pasta local do projeto
# ============================================================

from pathlib import Path
import requests

# Pasta local dos NDVI
pasta_ndvi = Path(
    r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao"
) / "data" / "processed" / "indices" / "ndvi"

# Cria a pasta caso não exista
pasta_ndvi.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("EXPORTAÇÃO DOS NDVI — ARQUIVOS LOCAIS")
print("=" * 60)

arquivos_baixados = []

for item in ndvi_mensais:

    data = item["data"].replace("-", "")

    nome = f"ndvi_{data}"
    arquivo_saida = pasta_ndvi / f"{nome}.tif"

    print(f"\nProcessando: {nome}")

    # URL de download direto do Earth Engine
    url = item["ndvi"].getDownloadURL({
        "name": nome,
        "scale": 10,
        "region": aoi_ee,
        "filePerBand": False,
        "format": "GEO_TIFF"
    })

    # Download
    resposta = requests.get(url)

    resposta.raise_for_status()

    with open(arquivo_saida, "wb") as arquivo:
        arquivo.write(resposta.content)

    arquivos_baixados.append(arquivo_saida)

    print(f"✓ Salvo: {arquivo_saida.name}")


print("\n" + "=" * 60)
print("EXPORTAÇÃO CONCLUÍDA")
print("=" * 60)

print(f"\n✓ Total de arquivos: {len(arquivos_baixados)}")
print(f"✓ Pasta:")
print(pasta_ndvi)

EXPORTAÇÃO DOS NDVI — ARQUIVOS LOCAIS

Processando: ndvi_20250108
✓ Salvo: ndvi_20250108.tif

Processando: ndvi_20250217
✓ Salvo: ndvi_20250217.tif

Processando: ndvi_20250309
✓ Salvo: ndvi_20250309.tif

Processando: ndvi_20250408
✓ Salvo: ndvi_20250408.tif

Processando: ndvi_20250513
✓ Salvo: ndvi_20250513.tif

Processando: ndvi_20250617
✓ Salvo: ndvi_20250617.tif

Processando: ndvi_20250727
✓ Salvo: ndvi_20250727.tif

Processando: ndvi_20250801
✓ Salvo: ndvi_20250801.tif

Processando: ndvi_20250915
✓ Salvo: ndvi_20250915.tif

Processando: ndvi_20251005
✓ Salvo: ndvi_20251005.tif

Processando: ndvi_20251121
✓ Salvo: ndvi_20251121.tif

Processando: ndvi_20251204
✓ Salvo: ndvi_20251204.tif

EXPORTAÇÃO CONCLUÍDA

✓ Total de arquivos: 12
✓ Pasta:
C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\indices\ndvi
